# Progetto Data Science — Customer Churn
## Previsione dell'abbandono clienti (Telco)

**Corso:** Introduzione al Pensiero Computazionale e alla Data Science — Prof. Francesco Poggi
**Dataset:** Customer_Churn
**Team:** Alessia, Michelle
**Obiettivo:** prevedere se un cliente abbandonerà il servizio (variabile target `Churn`), individuando i fattori più associati all'abbandono.

Questo notebook copre la **Fase 2 - Descrizione e comprensione del dataset**.

*Nota sull'uso di assistenti LLM: questo notebook è stato realizzato con l'assistenza di un LLM (Claude) come supporto alla programmazione (scrittura/formattazione del codice). Le ipotesi, le interpretazioni statistiche e le riflessioni critiche sono state elaborate e verificate insieme, comprendendo ogni passaggio, come richiesto dalle linee guida del corso.*


## 1. Import delle librerie e caricamento dati

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
sns.set_style('whitegrid')

df = pd.read_csv('../data/Customer_Churn.csv')
print(f"Dimensione del dataset: {df.shape[0]} righe, {df.shape[1]} colonne")
df.head()


Dimensione del dataset: 7043 righe, 21 colonne


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Struttura del dataset

Osserviamo i tipi di dato di ciascuna colonna. Il dataset descrive clienti di un'azienda di telecomunicazioni: dati demografici, servizi sottoscritti, informazioni contrattuali e la variabile target `Churn`.


In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Variabili numeriche ({len(num_cols)}): {num_cols}")
print()
print(f"Variabili categoriche/testuali ({len(cat_cols)}): {cat_cols}")


Variabili numeriche (3): ['SeniorCitizen', 'tenure', 'MonthlyCharges']

Variabili categoriche/testuali (18): ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


/tmp/ipykernel_6939/333414478.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


## 3. Qualità dei dati

### 3.1 Un problema nascosto: `TotalCharges`

`TotalCharges` dovrebbe essere una variabile numerica (importo totale fatturato), ma pandas la legge come stringa (`object`). Controlliamo perché.


In [4]:
print("Tipo attuale di TotalCharges:", df['TotalCharges'].dtype)

# Proviamo la conversione forzata a numerico, segnalando gli errori
total_charges_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_problematic = total_charges_numeric.isna().sum()
print(f"Righe che NON si convertono in numero: {n_problematic}")

df.loc[total_charges_numeric.isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]


Tipo attuale di TotalCharges: str
Righe che NON si convertono in numero: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


**Cosa sta succedendo:** le 11 righe problematiche hanno `TotalCharges` uguale a una stringa vuota/spazio, non un vero valore mancante segnalato da pandas. Controlliamo se c'è un pattern comune.


In [5]:
problematic_idx = total_charges_numeric.isna()
print("Valori di 'tenure' per le righe problematiche:")
print(df.loc[problematic_idx, 'tenure'].unique())
print()
print("Valori di 'Churn' per le righe problematiche:")
print(df.loc[problematic_idx, 'Churn'].value_counts())


Valori di 'tenure' per le righe problematiche:
[0]

Valori di 'Churn' per le righe problematiche:
Churn
No    11
Name: count, dtype: int64


**Interpretazione:** tutte le 11 righe con `TotalCharges` vuoto hanno `tenure = 0`, cioè sono clienti appena iscritti che non hanno ancora ricevuto una fattura — non è un dato mancante casuale, ma un caso sistematico e spiegabile. Inoltre nessuno di questi clienti ha abbandonato (`Churn = No`), il che è coerente: sono troppo recenti per aver potuto disdire. Convertiamo la colonna in numerico e decidiamo di impostare `TotalCharges = 0` per questi casi, dato che coincide con `tenure = 0` (nessuna fattura ancora emessa).


In [6]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Nuovo tipo:", df['TotalCharges'].dtype)
print("Valori mancanti residui:", df['TotalCharges'].isnull().sum())


Nuovo tipo: float64
Valori mancanti residui: 0


### 3.2 Altri controlli di qualità


In [7]:
# Missing value "ufficiali" (NaN) su tutte le colonne, dopo la pulizia
missing = df.isnull().sum()
print("Valori mancanti per colonna:")
print(missing[missing > 0] if missing.sum() > 0 else "Nessun valore mancante.")


Valori mancanti per colonna:
Nessun valore mancante.


In [8]:
# customerID è davvero un identificativo univoco?
print("customerID è univoco per ogni riga?", df['customerID'].is_unique)
print("Numero di valori distinti:", df['customerID'].nunique(), "su", len(df), "righe")


customerID è univoco per ogni riga? True
Numero di valori distinti: 7043 su 7043 righe


In [9]:
# Colonne costanti (un solo valore in tutto il dataset) -- da verificare, non presenti qui ma controlliamo comunque
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Colonne costanti:", constant_cols if constant_cols else "nessuna")


Colonne costanti: nessuna


## 4. Statistiche descrittive

Almeno 5 statistiche descrittive rilevanti per capire il dataset.


### 4.1 Distribuzione della variabile target (Churn)

In [10]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    'conteggio': churn_counts,
    'percentuale (%)': churn_pct.round(1)
})
summary_target


,conteggio,percentuale (%)
Churn,,
No,5174,73.5
Yes,1869,26.5


### 4.2 Statistiche su tenure, MonthlyCharges e TotalCharges

In [11]:
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)


,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


### 4.3 Distribuzione dei clienti per tipo di contratto e metodo di pagamento

In [12]:
print("Per Contract:")
print(df['Contract'].value_counts())
print()
print("Per PaymentMethod:")
print(df['PaymentMethod'].value_counts())


Per Contract:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

Per PaymentMethod:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64


### 4.4 Distribuzione dei clienti per servizio internet

In [13]:
internet_summary = df['InternetService'].value_counts()
internet_pct = df['InternetService'].value_counts(normalize=True) * 100

pd.DataFrame({'conteggio': internet_summary, 'percentuale (%)': internet_pct.round(1)})


,conteggio,percentuale (%)
InternetService,,
Fiber optic,3096,44.0
DSL,2421,34.4
No,1526,21.7


### 4.5 Composizione demografica: senior citizen, partner, dependents

In [14]:
demo_summary = pd.DataFrame({
    'Senior Citizen (%)': [df['SeniorCitizen'].mean() * 100],
    'Con Partner (%)': [(df['Partner'] == 'Yes').mean() * 100],
    'Con Dependents (%)': [(df['Dependents'] == 'Yes').mean() * 100],
}).round(1)
demo_summary


,Senior Citizen (%),Con Partner (%),Con Dependents (%)
0,16.2,48.3,30.0


## 5. Domande e ipotesi

Formuliamo almeno 8 domande/ipotesi che guideranno l'analisi esplorativa (Fase 3) e la modellazione (Fase 4).

1. I clienti con contratto mensile (`Contract = Month-to-month`) abbandonano più spesso rispetto a chi ha un contratto annuale o biennale.
2. I clienti con `tenure` basso (clienti recenti) hanno un tasso di churn più alto rispetto ai clienti di lunga data.
3. I clienti che pagano tramite `Electronic check` abbandonano più spesso rispetto agli altri metodi di pagamento.
4. I clienti con connessione internet in fibra ottica (`InternetService = Fiber optic`) hanno un tasso di churn più alto rispetto a DSL o nessuna connessione.
5. I clienti privi di servizi aggiuntivi come `OnlineSecurity` o `TechSupport` abbandonano più spesso rispetto a chi li ha attivati.
6. I clienti senior (`SeniorCitizen = 1`) hanno un tasso di churn diverso rispetto ai clienti non senior.
7. Un `MonthlyCharges` più alto è associato a una maggiore probabilità di abbandono.
8. I clienti senza partner e senza persone a carico (`Partner = No`, `Dependents = No`) abbandonano più frequentemente, essendo meno "radicati" nel servizio.
9. La fatturazione elettronica (`PaperlessBilling = Yes`) è associata a un tasso di churn diverso rispetto alla fatturazione tradizionale.

Nella Fase 3 verificheremo alcune di queste ipotesi con grafici e confronti tra gruppi.


## 6. Riflessioni critiche sui dati

**1. `TotalCharges` non era realmente numerico.** Come mostrato nella sezione 3.1, la colonna conteneva 11 valori vuoti mascherati da spazi, che pandas non riconosceva automaticamente come mancanti. Non si trattava di un errore casuale: tutti i casi corrispondevano a clienti con `tenure = 0`, cioè appena iscritti e non ancora fatturati. È un esempio concreto di come un dataset "senza valori mancanti" secondo un controllo superficiale (`isnull().sum()`) possa in realtà nasconderne, e di come vada sempre verificato il tipo di dato dichiarato da pandas rispetto al contenuto reale.

**2. Class imbalance moderato.** La variabile target è sbilanciata (~73.5% `No` contro ~26.5% `Yes`): non estremo come in altri dataset, ma comunque sufficiente perché l'accuracy da sola non sia una metrica affidabile. Andranno considerate precision, recall e f1-score, specialmente sulla classe minoritaria (`Churn = Yes`), che è quella di interesse pratico per l'azienda.

**3. Ridondanza nelle categorie "No internet service" / "No phone service".** Variabili come `OnlineSecurity`, `TechSupport`, `StreamingTV` includono, oltre a "Yes"/"No", anche il valore "No internet service" — che è in realtà un'informazione già contenuta in `InternetService`. Questo va tenuto presente nella fase di codifica delle variabili categoriche (Fase 4), perché rischia di introdurre ridondanza tra feature.

**4. `customerID` non è una feature.** È un identificativo univoco per ogni cliente (nessun duplicato), utile per tracciabilità ma privo di significato predittivo: va escluso dalle variabili usate per la modellazione, altrimenti rischia di comportarsi come un ID che il modello potrebbe usare in modo scorretto (overfitting).
